# `APIDataSource` - Acquisition de données depuis des APIs REST

**Module :** `kadi.kidas.sources.api_source`  
**Classe :** `APIDataSource`

---

## Introduction

`APIDataSource` gère les appels aux APIs agricoles et climatiques avec :

- **Rate limiting** : respect automatique d'un seuil de requêtes/seconde
- **Réessais avec backoff exponentiel** : en cas d'erreur 429, 500, 502, 503, 504
- **Détection automatique de la structure de réponse** : `data`, `results`, `items`, `features`
- **Authentification Bearer** : pour les APIs privées (MAEP, FAO API)

APIs compatibles : Open-Meteo, WFP VAM, FAO GIEWS, SoilGrids REST, MAEP Bénin

## 1. Importation et initialisation

```python
APIDataSource(
    api_url: str,
    auth_token: str | None = None,
    rate_limit_per_sec: float = 5.0,
)
```

In [2]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

from kadi.kidas.sources.api_source import APIDataSource

# API publique (pas d'authentification requise)
source_public = APIDataSource(
    api_url='https://api.open-meteo.com/v1/forecast',
    rate_limit_per_sec=10.0,
)
print(repr(source_public))

APIDataSource(source_path='https://api.open-meteo.com/v1/forecast', source_type='api', encoding='utf-8')


#### Ce que produit cette cellule

```
APIDataSource(source_path='https://api.open-meteo.com/v1/forecast', source_type='api', encoding='utf-8')
```

La représentation de l'objet montre trois champs :

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `https://api.open-meteo.com/v1/forecast` | L'URL de base de l'API. Pour une `APIDataSource`, le chemin est une URL et non un fichier. |
| `source_type` | `api` | Le type de source, toujours `api` pour cette classe. |
| `encoding` | `utf-8` | Les réponses REST JSON sont en UTF-8 par convention (RFC 7159). |

Open-Meteo est une API météorologique gratuite et sans clé d'API. Elle fournit des prévisions et des données historiques de température, précipitations, vent, etc. pour n'importe quel point GPS du globe. C'est l'une des rares APIs climatiques réellement utilisables sans abonnement dans un contexte de recherche agricole.

In [ ]:
# API avec authentification (FAO, MAEP, etc.)
source_prive = APIDataSource(
    api_url='https://api.maep.benin.gov/v1/prix', # Constaté actullement non fonctionnel
    auth_token='votre_token_api_secret',
    rate_limit_per_sec=2.0,
)
print(f"Authentification requise : {source_prive.get_metadata()['requires_auth']}")

Authentification requise : True


#### Ce que produit cette cellule

```
Authentification requise : True
```

La source est configurée pour une API privée qui demande un token d'authentification. La métadonnée `requires_auth` est déduite automatiquement : si `auth_token` est fourni lors de la construction, ce champ vaut `True`.

**Paramètres utilisés :**

| Paramètre | Valeur | Ce que ça signifie |
|---|---|---|
| `api_url` | `https://api.maep.benin.gov/v1/prix` | URL de l'API du MAEP (Ministère de l'Agriculture, de l'Elevage et de la Pêche du Bénin). Actuellement non fonctionnelle. |
| `auth_token` | `'votre_token_api_secret'` | Jeton d'authentification transmis dans l'en-tête HTTP `Authorization: Bearer ...` à chaque requête. |
| `rate_limit_per_sec` | `2.0` | Maximum 2 requêtes par seconde vers cette API, pour respecter la politique d'usage. |

Note : cette cellule n'a pas été exécutée dans le notebook (`execution_count: null`). Le résultat affiché est la sortie attendue, non une sortie réelle d'exécution.

## 2. `validate_connection()` - Tester la connectivité de l'API

In [4]:
# Test de connectivité sur une API réelle publique
source = APIDataSource('https://api.open-meteo.com/v1/forecast')
est_accessible = source.validate_connection()
print(f"Open-Meteo accessible : {est_accessible}")

Open-Meteo accessible : True


#### Ce que produit cette cellule

```
Open-Meteo accessible : True
```

`validate_connection()` envoie une requête **HTTP HEAD** à l'URL de l'API. Contrairement à GET, HEAD ne télécharge pas le corps de la réponse : il demande uniquement les en-têtes, ce qui est plus rapide et moins coûteux pour un simple test de disponibilité.

**Interprétation :**

| Résultat | Signification |
|---|---|
| `True` | Le serveur a répondu avec un code HTTP 2xx (ex : 200 OK). L'API est joignable. |
| `False` | Le serveur n'a pas répondu, ou a retourné une erreur de connexion (timeout, DNS introuvable). |

Cette méthode nécessite une connexion internet active. Si l'environnement d'exécution est isolé (CI/CD sans accès réseau, Jupyter hors ligne), elle retourne `False`.

**Différence avec les autres sources :** Pour `CSVDataSource` ou `ExcelDataSource`, `validate_connection()` vérifie l'existence d'un fichier sur le disque. Pour `APIDataSource`, elle teste la joignabilité réseau d'un endpoint distant.

## 3. `fetch_with_retry()` - Requête GET avec réessais automatiques

In [5]:
# Appel à l'API Open-Meteo (météo Cotonou, données gratuites)
source_meteo = APIDataSource(
    'https://api.open-meteo.com/v1/forecast',
    rate_limit_per_sec=5.0,
)

parametres = {
    'latitude': 6.36,
    'longitude': 2.42,
    'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
    'timezone': 'Africa/Abidjan',
    'forecast_days': 7,
}

try:
    reponse_brute = source_meteo.fetch_with_retry(parametres, max_retries=2)
    print(f"Clés de réponse : {list(reponse_brute.keys())}")
    if 'daily' in reponse_brute:
        print(f"Données 'daily' : {list(reponse_brute['daily'].keys())}")
except Exception as e:
    print(f"Connexion non disponible dans cet environnement : {e}")

Clés de réponse : ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
Données 'daily' : ['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']


#### Ce que produit cette cellule

```
Clés de réponse : ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds',
                   'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
Données 'daily' : ['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']
```

`fetch_with_retry()` exécute la requête GET avec les paramètres fournis et retourne le JSON brut (un dictionnaire Python) sans transformation.

**Analyse des clés de premier niveau :**

| Clé | Type | Contenu |
|---|---|---|
| `latitude` / `longitude` | float | Coordonnées GPS ajustées à la grille de la résolution du modèle (6.36 N, 2.42 E pour Cotonou). |
| `generationtime_ms` | float | Temps de génération de la réponse côté serveur, en millisecondes. |
| `utc_offset_seconds` | int | Décalage horaire en secondes. `0` car le fuseau `Africa/Abidjan` est UTC+0 (même méridien que Greenwich). |
| `timezone` | str | Fuseau horaire demandé : `Africa/Abidjan`. Le Bénin est en UTC+1 mais le Togo voisin utilise `Africa/Abidjan` (UTC+0). |
| `timezone_abbreviation` | str | Abréviation du fuseau : `GMT`. |
| `elevation` | float | Altitude du point GPS en mètres au-dessus du niveau de la mer. |
| `daily_units` | dict | Unités des variables : `°C` pour les températures, `mm` pour les précipitations. |
| `daily` | dict | Les données journalières elles-mêmes : 4 clés, une par variable demandée. |

**Contenu de `daily` :**

| Clé | Contenu |
|---|---|
| `time` | Liste des 7 dates en format ISO (ex : `['2024-01-15', ..., '2024-01-21']`). |
| `temperature_2m_max` | Température maximale journalière à 2 m du sol, en °C. |
| `temperature_2m_min` | Température minimale journalière, en °C. |
| `precipitation_sum` | Total de précipitation journalier, en mm. |

**Mécanisme de réessai :** Si le serveur répond avec un code 429 (trop de requêtes) ou 5xx (erreur serveur), `fetch_with_retry()` attend un délai croissant avant de retenter. Avec `max_retries=2`, la requête peut être tentée jusqu'à 3 fois au total.

## 4. `read()` - Lecture et conversion en DataFrame

`read()` appelle `fetch_with_retry()` et convertit automatiquement la réponse JSON en DataFrame. Il détecte les clés standard : `data`, `results`, `items`, `records`, `features`.

In [6]:
# Démonstration avec mock (simulation d'une réponse API)
from unittest.mock import patch, MagicMock

# Simulation d'une réponse WFP VAM (prix des marchés)
reponse_simulee = [
    {"market": "Dantokpa", "commodity": "Maize", "price": 350, "date": "2024-01-15"},
    {"market": "Parakou",  "commodity": "Cowpea", "price": 500, "date": "2024-01-15"},
    {"market": "Bohicon",  "commodity": "Yam",    "price": 250, "date": "2024-01-15"},
]

mock_response = MagicMock()
mock_response.status_code = 200
mock_response.json.return_value = reponse_simulee
mock_response.raise_for_status.return_value = None

source = APIDataSource('https://api.wfp.org/vam/prices')
with patch('kadi.kidas.sources.api_source.requests.get', return_value=mock_response):
    df = source.read(params={'country': 'BEN', 'commodity': 'Maize'})

print(f"DataFrame chargé depuis l'API : {len(df)} lignes × {len(df.columns)} colonnes")
df

DataFrame chargé depuis l'API : 3 lignes × 4 colonnes


,market,commodity,price,date
0,Dantokpa,Maize,350,2024-01-15
1,Parakou,Cowpea,500,2024-01-15
2,Bohicon,Yam,250,2024-01-15


#### Ce que produit cette cellule

```
DataFrame chargé depuis l'API : 3 lignes × 4 colonnes
```

Suivi du tableau avec 3 lignes de données de prix de marché.

**Technique utilisée : mock**

Cette cellule n'effectue pas de vraie requête réseau. Elle utilise `unittest.mock.patch` pour **remplacer temporairement** `requests.get` par un objet simulé (`MagicMock`) qui retourne une réponse prédéfinie.

```
requests.get(...) → intercepté → retourne mock_response
mock_response.status_code = 200
mock_response.json() = [{'market': 'Dantokpa', ...}, ...]
```

**Pourquoi une liste directe retourne 3 lignes ?**

La réponse simulée est une **liste de dictionnaires** (pas un dict avec une clé `data`). `read()` détecte ce cas : quand la réponse est directement une liste, elle la convertit immédiatement en DataFrame sans chercher de clé conteneur. Résultat : 3 lignes (un enregistrement par marché) et 4 colonnes.

**Contenu du tableau :**

| Marché | Produit | Prix (XOF) | Contexte |
|---|---|---|---|
| Dantokpa | Maize (maïs) | 350 | Grand marché de Cotonou, référence nationale des prix céréaliers. |
| Parakou | Cowpea (niébé) | 500 | Capitale du nord, marché de transit vers le Niger et le Burkina. |
| Bohicon | Yam (igname) | 250 | Carrefour commercial du centre, important pour les tubercules. |

In [7]:
# Démonstration : réponse avec clé 'data' (structure FAO)
reponse_fao = {
    "meta": {"total": 3},
    "data": [
        {"country": "Benin", "crop": "maize", "production_tonnes": 1200000},
        {"country": "Benin", "crop": "cassava", "production_tonnes": 3800000},
        {"country": "Benin", "crop": "yam",    "production_tonnes": 2900000},
    ],
}

mock_fao = MagicMock()
mock_fao.status_code = 200
mock_fao.json.return_value = reponse_fao
mock_fao.raise_for_status.return_value = None

source_fao = APIDataSource('https://api.fao.org/faostat/crops')
with patch('kadi.kidas.sources.api_source.requests.get', return_value=mock_fao):
    df_fao = source_fao.read({'area': 'BEN', 'year': 2024})

print("Production agricole Bénin (FAO) :")
df_fao

Production agricole Bénin (FAO) :


,country,crop,production_tonnes
0,Benin,maize,1200000
1,Benin,cassava,3800000
2,Benin,yam,2900000


#### Ce que produit cette cellule

```
Production agricole Bénin (FAO) :
```

Suivi du tableau avec 3 lignes de données de production.

**Différence clé avec la cellule précédente : détection de la clé `data`**

Cette fois, la réponse simulée est un **dictionnaire** avec deux clés de premier niveau : `meta` et `data`. `read()` détecte automatiquement la clé `data` et extrait uniquement cette liste pour construire le DataFrame, en ignorant `meta`.

```
Réponse brute : {'meta': {...}, 'data': [...]}
                                    ^
                    read() extrait cette liste
```

**Clés détectées automatiquement par `read()` :**

| Clé | Utilisée par |
|---|---|
| `data` | FAO GIEWS, WFP VAM, MAEP |
| `results` | APIs REST standards |
| `items` | APIs de type catalogue |
| `records` | Exports Airtable, Notion |
| `features` | GeoJSON (coordonnées spatiales) |

**Données de production affichées :**

| Culture | Production (tonnes) | Rang Bénin |
|---|---|---|
| Maize (maïs) | 1 200 000 | Première céréale du pays. |
| Cassava (manioc) | 3 800 000 | Première culture vivrière absolue, base de l'alimentation. |
| Yam (igname) | 2 900 000 | Deuxième tubercule, très présent dans les régions Centre et Nord. |

## 5. Simulation du mécanisme de réessai (backoff exponentiel)

In [15]:
# Démonstration du backoff : 1ère tentative → 503, 2ème → 200
mock_503 = MagicMock()
mock_503.status_code = 503  # Service Unavailable → réessayable

mock_200 = MagicMock()
mock_200.status_code = 200
mock_200.json.return_value = [{"culture": "maïs", "prix": 350}]
mock_200.raise_for_status.return_value = None

source = APIDataSource('https://api.example.com/prix', rate_limit_per_sec=100)

with patch('kadi.kidas.sources.api_source.requests.get',
           side_effect=[mock_503, mock_200]):
    resultat = source.fetch_with_retry({}, max_retries=2, backoff_sec=0.01)

print(f"Résultat après retry : {resultat}")

Erreur HTTP 503. Réessai dans 0.0 secondes (tentative 1/2).


Résultat après retry : [{'culture': 'maïs', 'prix': 350}]


#### Ce que produit cette cellule

```
[erreur standard] Erreur HTTP 503. Réessai dans 0.0 secondes (tentative 1/2).
[sortie standard] Résultat après retry : [{'culture': 'maïs', 'prix': 350}]
```

Cette cellule simule une panne temporaire du serveur et montre que `fetch_with_retry()` récupère correctement la situation.

**Déroulement chronologique :**

| Tentative | Réponse simulée | Comportement |
|---|---|---|
| 1 | `503 Service Unavailable` | `fetch_with_retry()` détecte l'erreur 503 (réessayable), logue un avertissement sur l'erreur standard, attend `backoff_sec=0.01` seconde. |
| 2 | `200 OK` | La requête réussit. Le JSON `[{'culture': 'maïs', 'prix': 350}]` est retourné. |

**Message d'avertissement :**

```
Erreur HTTP 503. Réessai dans 0.0 secondes (tentative 1/2).
```

- `0.0 secondes` : le délai d'attente est quasi-nul ici (`backoff_sec=0.01`) pour accélérer la démonstration. En production, le backoff exponentiel calcule un délai croissant : `0.01s`, `0.02s`, `0.04s`, etc.
- `tentative 1/2` : c'est le premier réessai sur 2 maximum autorisés.

**Codes HTTP réessayables vs non réessayables :**

| Code | Signification | Réessayable |
|---|---|---|
| `429` | Trop de requêtes (rate limit) | Oui |
| `500` | Erreur interne serveur | Oui |
| `502` | Bad Gateway | Oui |
| `503` | Service indisponible | Oui |
| `504` | Gateway Timeout | Oui |
| `404` | Endpoint introuvable | Non (inutile de réessayer) |
| `401` | Non autorisé | Non (token invalide) |

## 6. `write()` - Envoi de données vers une API (POST)

In [9]:
import pandas as pd

df_a_envoyer = pd.DataFrame({
    'culture': ['maïs', 'niébé'],
    'prix_xof': [350, 500],
    'marche': ['Dantokpa', 'Parakou'],
})

# Simulation d'une réponse POST réussie
mock_post = MagicMock()
mock_post.status_code = 201  # Created
mock_post.raise_for_status.return_value = None

source = APIDataSource('https://api.maep.benin.gov/v1/prix')
with patch('kadi.kidas.sources.api_source.requests.post', return_value=mock_post):
    succes = source.write(df_a_envoyer)

print(f"Envoi POST réussi : {succes}")

Envoi POST réussi : True


#### Ce que produit cette cellule

```
Envoi POST réussi : True
```

`write()` sérialise le DataFrame en JSON et l'envoie à l'API via une requête HTTP **POST**.

**Déroulement :**

1. Le DataFrame est converti en liste de dictionnaires (format `records`).
2. La liste est envoyée dans le corps de la requête POST avec l'en-tête `Content-Type: application/json`.
3. Le mock simule une réponse `201 Created` : le serveur a bien enregistré les données.
4. `write()` retourne `True` si le code de réponse indique un succès (2xx).

**Données envoyées :**

```json
[
  {"culture": "maïs",  "prix_xof": 350, "marche": "Dantokpa"},
  {"culture": "niébé", "prix_xof": 500, "marche": "Parakou"}
]
```

**Code HTTP 201 vs 200 :**

| Code | Signification | Contexte |
|---|---|---|
| `200 OK` | Succès générique | Lecture (GET), mise à jour (PUT). |
| `201 Created` | Ressource créée | Création d'un nouvel enregistrement (POST). |

L'API MAEP Bénin (`api.maep.benin.gov`) est l'endpoint officiel du Ministère de l'Agriculture pour la collecte des prix de marché. Elle est actuellement non fonctionnelle en accès public, d'où l'utilisation d'un mock pour la démonstration.

## 7. `get_schema()` et `get_metadata()`

In [10]:
source = APIDataSource(
    'https://api.open-meteo.com/v1/forecast',
    rate_limit_per_sec=10.0,
)

print("Schéma de la source :")
schema = source.get_schema()
for cle, val in schema.items():
    print(f"  {cle:25s} : {val}")

print("\nMétadonnées complètes :")
meta = source.get_metadata()
for cle, val in meta.items():
    print(f"  {cle:25s} : {val}")

Schéma de la source :
  api_url                   : https://api.open-meteo.com/v1/forecast
  requires_auth             : False
  rate_limit_per_sec        : 10.0
  last_read                 : None

Métadonnées complètes :
  source_path               : https://api.open-meteo.com/v1/forecast
  source_type               : api
  requires_auth             : False
  rate_limit_per_sec        : 10.0
  last_read                 : None


#### Ce que produit cette cellule

```
Schéma de la source :
  api_url                   : https://api.open-meteo.com/v1/forecast
  requires_auth             : False
  rate_limit_per_sec        : 10.0
  last_read                 : None

Métadonnées complètes :
  source_path               : https://api.open-meteo.com/v1/forecast
  source_type               : api
  requires_auth             : False
  rate_limit_per_sec        : 10.0
  last_read                 : None
```

Deux méthodes sont appelées sur la même instance. Leurs sorties se ressemblent beaucoup, mais elles ont des rôles distincts.

**`get_schema()` — Configuration de la connexion**

| Champ | Valeur | Ce que ça signifie |
|---|---|---|
| `api_url` | `https://api.open-meteo.com/v1/forecast` | URL de base de l'API. |
| `requires_auth` | `False` | Aucun token fourni, l'API est publique. |
| `rate_limit_per_sec` | `10.0` | Maximum 10 requêtes par seconde. |
| `last_read` | `None` | Aucune lecture effectuée sur cette instance. |

**`get_metadata()` — Métadonnées complètes**

Retourne les mêmes champs, mais avec deux champs supplémentaires en tête :

| Champ ajouté | Valeur | Ce que ça signifie |
|---|---|---|
| `source_path` | `https://api.open-meteo.com/v1/forecast` | Champ commun à toutes les sources kidas (CSV, Excel, JSON, NetCDF, API). |
| `source_type` | `api` | Type de source, commun à toutes les sources. |

**Différence entre les deux méthodes :**

`get_schema()` est spécifique aux APIs (URL, auth, rate) tandis que `get_metadata()` est l'interface standard commune à toutes les sources, incluant les champs partagés `source_path` et `source_type`. C'est `get_metadata()` qui est utilisé par `DataPipeline` pour générer le rapport de traçabilité.

## 8. Cas d'usage réel : Open-Meteo (API gratuite)

In [16]:
# Tentative d'appel réel à Open-Meteo (nécessite une connexion internet)
import requests

source_openmeteo = APIDataSource(
    'https://api.open-meteo.com/v1/forecast',
    rate_limit_per_sec=1.0,
)

try:
    df_meteo = source_openmeteo.read(params={
        'latitude': 6.36,       # Cotonou
        'longitude': 2.42,
        'daily': 'precipitation_sum,temperature_2m_max',
        'timezone': 'Africa/Abidjan',
        'forecast_days': 7,
    })
    print(f"Données météo Cotonou ({len(df_meteo)} jours) :")
    print(df_meteo)
except Exception as e:
    print(f"Note : connexion non disponible dans cet environnement.\nErreur : {e}")

Données météo Cotonou (1 jours) :
   latitude  longitude  generationtime_ms  utc_offset_seconds        timezone  \
0  6.362039    2.41206           0.064731                   0  Africa/Abidjan   

  timezone_abbreviation  elevation daily_units.time  \
0                   GMT        8.0          iso8601   

  daily_units.precipitation_sum daily_units.temperature_2m_max  \
0                            mm                             °C   

                                          daily.time  \
0  [2026-07-07, 2026-07-08, 2026-07-09, 2026-07-1...   

                  daily.precipitation_sum  \
0  [3.6, 3.7, 37.6, 34.7, 2.7, 9.2, 21.2]   

                     daily.temperature_2m_max  
0  [28.8, 27.9, 27.8, 26.7, 27.8, 27.5, 26.7]  


#### Ce que produit cette cellule

```
Données météo Cotonou (1 jours) :
```

Suivi d'un tableau très large avec **1 seule ligne** et de nombreuses colonnes aux noms à notation pointée.

**Pourquoi "1 jours" et non "7 jours" ?**

C'est le comportement attendu, non une erreur. La réponse d'Open-Meteo est un **dictionnaire imbriqué** (pas une liste) :

```json
{
  "latitude": 6.362039,
  "longitude": 2.41206,
  "daily": {
    "time": ["2026-07-07", "2026-07-08", ...],
    "precipitation_sum": [3.6, 3.7, 37.6, ...],
    "temperature_2m_max": [28.8, 27.9, ...]
  }
}
```

`read()` ne détecte aucune clé standard (`data`, `results`, etc.) dans la réponse d'Open-Meteo. Il applique alors l'aplatissement complet (`flatten_json`) sur le dictionnaire entier : **1 ligne** avec toutes les valeurs terminales comme colonnes.

**Colonnes produites et leur contenu :**

| Colonne | Valeur | Type |
|---|---|---|
| `latitude` | `6.362039` | Coordonnée ajustée à la grille du modèle. |
| `longitude` | `2.41206` | Idem. |
| `elevation` | `8.0` | Altitude de Cotonou en mètres (proche du niveau de la mer). |
| `timezone` | `Africa/Abidjan` | Fuseau UTC+0, identique pour le Togo et la Côte d'Ivoire. |
| `daily_units.precipitation_sum` | `mm` | Unité des précipitations : millimètres. |
| `daily_units.temperature_2m_max` | `°C` | Unité des températures : degrés Celsius. |
| `daily.time` | `['2026-07-07', ..., '2026-07-13']` | Les 7 dates sous forme de liste (stockée dans une seule cellule). |
| `daily.precipitation_sum` | `[3.6, 3.7, 37.6, 34.7, 2.7, 9.2, 21.2]` | Cumuls journaliers en mm sur 7 jours. Pics le 3e et 4e jour : fortes pluies prévisibles a Cotonou en juillet (saison des pluies). |
| `daily.temperature_2m_max` | `[28.8, 27.9, 27.8, 26.7, 27.8, 27.5, 26.7]` | Températures maximales autour de 27-28 °C, normales pour juillet a Cotonou. |

**Pour obtenir un DataFrame de 7 lignes (une par jour), il faut post-traiter :**

```python
import pandas as pd
daily = reponse_brute['daily']        # Dictionnaire des séries temporelles
df_jours = pd.DataFrame(daily)        # 7 lignes × 3 colonnes
```

Cette transformation manuelle est nécessaire car Open-Meteo n'utilise pas une clé standard détectable par `read()`.

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `validate_connection()` | Requête HEAD pour tester la connectivité |
| `fetch_with_retry(params, max_retries, backoff_sec)` | GET avec backoff exponentiel sur 429/5xx |
| `read(params)` | Appel GET + conversion DataFrame |
| `write(data)` | Envoi POST du DataFrame en JSON |
| `get_schema()` | Configuration de la source (URL, auth, rate) |
| `get_metadata()` | Métadonnées complètes avec last_read |